# 28장 · LangGraph의 메시지 리듀서 add_messages

리듀서는 상태에 새 값이 들어왔을 때 기존 값과 결합하는 규칙입니다. add_messages는 메시지 ID에 따라 새 메시지를 추가하거나 기존 메시지를 갱신합니다.

응답 노드가 새 AIMessage 한 건을 반환할 때 기본 갱신과 add_messages의 결과를 비교합니다. 메시지 목록의 길이와 ID를 보면 기존 발화가 보존됐는지 확인할 수 있습니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain-core 1.6.12, langgraph 1.2.16입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 기본 덮어쓰기와 add_messages의 누적

목록을 상태 필드에 넣는 것만으로 이전 목록 뒤에 붙지는 않습니다. PlainState에서는 노드가 돌려준 messages가 이전 값을 덮습니다. ChatState는 Annotated에 add_messages를 적어 이전 메시지와 새 메시지를 합치는 규칙을 정합니다.

### 셀 01 · 기본 상태 갱신의 덮어쓰기

제공 셀입니다. 두 상태 정의와 응답 함수는 그대로 두고 PlainState에서 돌아온 메시지 종류를 확인하세요.

In [1]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage, AnyMessage
from langgraph.graph.message import add_messages

class PlainState(TypedDict):
    messages: list[AnyMessage]

class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def respond(state: dict) -> dict:
    humans = [message for message in state["messages"] if isinstance(message, HumanMessage)]
    if not humans:
        raise ValueError("사용자 메시지가 필요합니다.")
    last = humans[-1]
    return {"messages": [AIMessage(content=f"확인한 질문: {last.content}", id="a1")]}

def make_graph(schema=ChatState):
    builder = StateGraph(schema)
    builder.add_node("respond", respond)
    builder.add_edge(START, "respond")
    builder.add_edge("respond", END)
    return builder.compile()

initial = {'messages': [HumanMessage(content='대출 기간은?', id='h1')]}
plain = make_graph(PlainState).invoke(initial)
print([m.type for m in plain['messages']])

['ai']


ai 하나가 남았습니다. respond가 사용자 질문은 읽었지만 AIMessage 한 개만 돌려줬기 때문입니다.

사용자가 실제로 질문했는데도 마지막 출력에는 ai만 남습니다. 응답 함수는 질문을 읽었고, 반환할 때에는 답 한 건을 담은 새 목록을 건넸습니다. 같은 반환값을 다음 셀의 add_messages가 받으면 h1과 a1이 함께 남습니다. 바뀐 곳은 응답 함수보다 상태를 갱신하는 규칙입니다.

### 문자열 목록과 메시지 객체

<p>HumanMessage는 사용자 발화, AIMessage는 모델 측 발화의 역할을 표현합니다. 이 예제는 LLM 없이도 AIMessage 객체를 생성하므로 객체 타입만 보고 실제 모델 호출이 있었다고 판단하지 않습니다. content는 내용, id는 메시지를 구별하고 수정할 때 쓰는 식별자입니다.</p>

<p>역할과 내용을 함께 보존하면 뒤에서 모델에 입력을 구성하거나 도구 결과를 다룰 때 어떤 발화인지 구분할 수 있습니다. 도구 호출 메시지까지 한꺼번에 추가하지 않고 이번에는 human과 ai 두 역할만 사용합니다.</p>

<table><caption>읽을 필드</caption><thead><tr><th scope="col">필드</th><th scope="col">예</th><th scope="col">해석</th></tr>
</thead><tbody><tr><td>type</td><td>human / ai</td><td>발화 역할</td></tr>
<tr><td>content</td><td>대출 기간은?</td><td>내용</td></tr>
<tr><td>id</td><td>h1 / a1</td><td>메시지 식별자</td></tr>
</tbody></table>

### 리듀서의 리스트 결합 규칙

<p>PlainState의 messages는 새 리스트를 반환하면 이전 값이 교체됩니다. ChatState의 Annotated 표기는 messages의 타입과 함께 add_messages 갱신 규칙을 지정합니다. Python의 Annotated가 스스로 저장하거나 대화를 이어 주는 것은 아닙니다. LangGraph가 그 규칙을 읽어 상태 업데이트에 적용합니다.</p>

<p>add_messages는 새 ID를 가진 메시지를 추가하고 기존 ID와 같은 메시지는 해당 위치의 내용을 갱신합니다. 문자열이 같은지 비교해 중복을 제거하는 기능으로 이해하면 안 됩니다. 단순 리스트 덧셈과도 다릅니다.</p>

### 셀 02 · add_messages로 메시지 누적

입력과 respond는 그대로 두고 스키마만 바꿉니다.

In [2]:
combined = make_graph(ChatState).invoke(initial)
print([(m.id, m.type) for m in combined['messages']])

[('h1', 'human'), ('a1', 'ai')]


h1 뒤에 a1이 보입니다. 서로 다른 ID를 받은 메시지는 순서대로 남습니다. 응답 함수가 이전 메시지까지 다시 보낼 필요가 없습니다.

### 새 응답 메시지의 부분 갱신

<p>respond는 현재 메시지 목록에서 마지막 사용자 메시지를 찾아 응답 한 개를 반환합니다. messages에는 새 응답만 들어 있습니다. ChatState의 갱신 규칙이 기존 사용자 발화와 이 변경분을 합칩니다.</p>

<p>상태의 목록을 직접 append한 뒤 전체 목록을 반환하는 방식은 어떤 변경분을 적용했는지 흐리게 만듭니다. 이 예제처럼 입력은 읽고 새 목록을 반환하면 단위 검사가 쉬워집니다. 응답 ID a1은 단일 응답 비교용 고정값입니다. 다중 응답에 그대로 재사용하는 정책은 아닙니다.</p>

## 동일한 메시지 ID로 내용 갱신

내용이 비슷해도 ID가 다르면 새 발화입니다. 반대로 ID를 그대로 쓰면 그 발화를 수정할 수 있습니다. 같은 답변 ID를 계속 재사용하면 새 대화가 쌓이지 않는 이유도 여기에 있습니다.

<!-- lesson-visual practice-space -->
**수정 답변의 ID를 바꾸면 새 턴이 생길까요?**

초기 목록에 h1과 a1이 있습니다. 같은 수정 답변을 보내면서 리듀서와 ID를 바꾸고 어느 칸이 남는지 보세요.

add_messages에서 같은 a1은 기존 칸을 고쳐 2개, 새 a2는 추가되어 3개입니다. 기본 덮어쓰기에서는 받은 a1만 남습니다.

[웹 교재에서 직접 조작하기](https://langchain-lecture.vercel.app/chapter28/#practice-space)
<!-- /lesson-visual practice-space -->

### 과제 03 · 메시지 ID로 기존 질문 교체

h1의 내용을 ‘반납 방법은?’으로 바꿔 보세요. 수정 후 사용자 메시지의 개수와 내용을 표시하세요.

이번에 수정하는 initial에는 사용자 메시지 h1 하나만 있습니다. 같은 ID로 고치면 한 자리의 내용만 바뀝니다. h1과 a1이 모두 있는 combined를 입력했다면 a1도 그대로 남습니다.

In [3]:
# add_messages에 기존 목록과 수정할 메시지 목록을 전달하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>changed = add_messages(initial[&#x27;messages&#x27;], [HumanMessage(id=&#x27;h1&#x27;, content=&#x27;반납 방법은?&#x27;)])
print(len(changed), changed[0].content)</code></pre>
<p>실행 결과</p><pre><code>1 반납 방법은?</code></pre>
<p>메시지는 여전히 한 개입니다. 두 개가 나온다면 수정할 메시지의 id가 h1인지 확인하세요.</p>
</details>

### 누락 발화 복원 테스트

<table><caption>오류와 확인</caption><thead><tr><th scope="col">문제</th><th scope="col">확인할 지점</th><th scope="col">조치</th></tr>
</thead><tbody><tr><td>질문이 사라짐</td><td>PlainState 여부</td><td>필요한 갱신 규칙 적용</td></tr>
<tr><td>동일 문장이 여러 개</td><td>ID가 서로 다른지</td><td>내용과 ID 의도를 따로 확인</td></tr>
<tr><td>이전 답변이 바뀜</td><td>응답 ID 재사용</td><td>수정이 아니라 새 답변이면 새 ID</td></tr>
<tr><td>사용자 메시지가 필요합니다</td><td>빈 목록·역할</td><td>HumanMessage 입력 확인</td></tr>
<tr><td>새 invoke에서 이전 기록 없음</td><td>체크포인터 없음</td><td>이 장은 자동 대화 지속을 구현하지 않음</td></tr>
</tbody></table>

## 돌아보기

아래 실습에서는 이미 답변을 받은 질문을 정정합니다. 질문의 ID를 유지하면 어떤 내용이 바뀌고 어떤 답이 그대로 남는지 두 메시지를 함께 읽어 보세요.

### 과제 04 · 같은 발화의 정정이 답변에 미치는 영향

combined 안의 질문 h1을 반납 질문으로 교체하세요. 기존 답변 a1의 내용도 함께 출력해 갱신 범위를 확인합니다.

In [ ]:
# add_messages에 combined["messages"]와 h1의 교정 내용을 전달하세요.

<details><summary>풀이와 실행 결과 보기</summary>
<pre><code>correction = HumanMessage(id="h1", content="반납 방법은?")
revised = add_messages(combined["messages"], [correction])
for message in revised:
    print(message.id, message.content)</code></pre>
<p>실행 결과</p><pre><code>h1 반납 방법은?
a1 확인한 질문: 대출 기간은?</code></pre>
<p>h1이 바뀌어도 이미 저장한 a1 문장은 대출 기간을 가리킵니다. 메시지 저장 규칙은 기존 답변을 다시 만드는 노드를 호출하지 않습니다.</p>
</details>